## Library downloads

In [ ]:
# Cell 1: Environment Setup
!pip install torch-geometric
!pip install rdkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 55.0 MB/s eta 0:00:00


In [ ]:
# ==========================================
# 1. Environment Setup & Imports
# ==========================================
import os
import warnings
import csv
import random
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import TransformerConv
from torch_geometric.utils import to_dense_batch
from torch.optim.lr_scheduler import ReduceLROnPlateau

from rdkit import Chem

warnings.filterwarnings("ignore")

# Configuration
CONFIG = {
    # --- Paths ---
    "base_path": "/content/drive/MyDrive/pmechrp_datasets/two_stage_models/mixed_splits",
    "num_folds": 5,

    # --- Hyperparameters ---
    "num_epochs": 50,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "hidden_dim": 128,
    "num_heads": 8,
    "num_layers": 4,
    "dropout": 0.1,
    "max_grad_norm": 1.0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "add_explicit_hydrogens": True,
    # "class_weights": [10.0, 1.0, 10.0] -> Epoch 23 | Loss: 0.003/0.005 | Rec: 0.900 | Pre: 0.308
    # "class_weights": [3.0, 1.0, 3.0]   ->  Ep 07 | Val Loss: 0.0025 | Rec: 0.720 | Pre: 0.347 | F1: 0.468 | Top3: 0.127
    "class_weights": [4.0, 1.0, 4.0]  # ->  Ep 20 | Val Loss: 0.0023 | Rec: 0.849 | Pre: 0.403 | F1: 0.547 | Top3: 0.812
}

## Data processing

In [ ]:
# ==========================================
# 2. Data Processing Pipeline
# ==========================================
def one_hot_encode(value, allowed_choices):
    encoding = [0] * (len(allowed_choices) + 1)
    if value in allowed_choices:
        encoding[allowed_choices.index(value)] = 1
    else:
        encoding[-1] = 1
    return encoding

def _align_atoms(mol):
    """Aligns atoms: Mapped atoms first (sorted), followed by unmapped atoms."""
    sorted_atoms = sorted(
        mol.GetAtoms(),
        key=lambda a: a.GetAtomMapNum() if a.GetAtomMapNum() > 0 else float('inf')
    )
    ordered_indices = [a.GetIdx() for a in sorted_atoms]
    return Chem.RenumberAtoms(mol, ordered_indices)

def _featurize_nodes(aligned_mol):
    ALLOWED_ELEMENTS = [1, 6, 7, 8, 9, 15, 16, 17, 35, 53]
    ALLOWED_HYBRIDIZATIONS = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2, Chem.rdchem.HybridizationType.SP3]
    ALLOWED_CHIRAL = [Chem.rdchem.ChiralType.CHI_UNSPECIFIED, Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW, Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW, Chem.rdchem.ChiralType.CHI_OTHER]

    node_features = []
    for atom in aligned_mol.GetAtoms():
        element_one_hot = one_hot_encode(atom.GetAtomicNum(), ALLOWED_ELEMENTS)
        num_hs = atom.GetTotalNumHs() if atom.GetNumRadicalElectrons() == 0 else 0
        hyb_one_hot = one_hot_encode(atom.GetHybridization(), ALLOWED_HYBRIDIZATIONS)
        chiral_one_hot = one_hot_encode(atom.GetChiralTag(), ALLOWED_CHIRAL)

        features = element_one_hot + [
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetIsAromatic()),
            num_hs,
            int(atom.IsInRing())
        ] + hyb_one_hot + chiral_one_hot
        node_features.append(features)

    return torch.tensor(node_features, dtype=torch.float)

def _featurize_edges(aligned_mol):
    edge_indices, edge_features = [], []
    allowed_bond_types = [1.0, 2.0, 3.0, 4.0]

    for bond in aligned_mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bond_type = 4.0 if bond.GetIsAromatic() else bond.GetBondTypeAsDouble()
        bond_one_hot = one_hot_encode(bond_type, allowed_bond_types)
        bond_feats = bond_one_hot + [int(bond.IsInRing())]

        edge_indices.extend([[i, j], [j, i]])
        edge_features.extend([bond_feats, bond_feats])

    if edge_indices:
        edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_features, dtype=torch.float)
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, len(allowed_bond_types) + 2), dtype=torch.float)

    return edge_index, edge_attr

def process_mapped_smiles(mapped_smiles, add_hs=True):
    mol = Chem.MolFromSmiles(mapped_smiles)
    if mol is None:
        raise ValueError("Invalid SMILES string")

    if add_hs:
        mol = Chem.AddHs(mol)

    aligned_mol = _align_atoms(mol)
    x = _featurize_nodes(aligned_mol)
    edge_index, edge_attr = _featurize_edges(aligned_mol)
    return aligned_mol, Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

def parse_pmech_row(raw_csv_string):
    # Use standard CSV parsing to cleanly handle commas inside the quoted arrow strings
    reader = csv.reader([str(raw_csv_string).strip()])
    parts = next(reader)

    if len(parts) < 2:
        raise ValueError(f"Malformed row (not enough columns): {raw_csv_string}")

    equation = parts[0]
    arrows = parts[1]

    equation_parts = equation.split(">>")
    if len(equation_parts) != 2:
        raise ValueError(f"Malformed equation: {equation}")

    return equation_parts[0], equation_parts[1], arrows

def generate_delta_from_arrows(arrow_string, mol):
    num_nodes = mol.GetNumAtoms()
    delta = torch.zeros((num_nodes, num_nodes), dtype=torch.long)
    map_to_idx = {atom.GetAtomMapNum(): atom.GetIdx() for atom in mol.GetAtoms() if atom.GetAtomMapNum() > 0}

    if not isinstance(arrow_string, str) or not arrow_string.strip():
        return delta

    for arrow in arrow_string.split(';'):
        arrow = arrow.strip().replace('*', '').replace(' ', '')
        if '=' not in arrow:
            continue

        src_s, dst_s = arrow.split('=')
        src_maps = [int(m) for m in src_s.split(',') if m.isdigit()]
        dst_maps = [int(m) for m in dst_s.split(',') if m.isdigit()]

        if len(src_maps) == 2:  # Bond Breaking
            u, v = src_maps[0], src_maps[1]
            if u in map_to_idx and v in map_to_idx:
                delta[map_to_idx[u], map_to_idx[v]] = -1
                delta[map_to_idx[v], map_to_idx[u]] = -1
        elif len(src_maps) == 1 and len(dst_maps) == 1:  # Bond Formation
            u, v = src_maps[0], dst_maps[0]
            if u in map_to_idx and v in map_to_idx:
                delta[map_to_idx[u], map_to_idx[v]] = 1
                delta[map_to_idx[v], map_to_idx[u]] = 1
                # To maintain symmetry in targets, ensure reverse is also formed
                delta[map_to_idx[v], map_to_idx[u]] = 1
        elif len(dst_maps) == 2:  # Concerted Shift
            u, v = dst_maps[0], dst_maps[1]
            if u in map_to_idx and v in map_to_idx:
                delta[map_to_idx[u], map_to_idx[v]] = 1
                delta[map_to_idx[v], map_to_idx[u]] = 1

    return delta



In [ ]:
# ==========================================
# 3. Dataset & Safe Collation
# ==========================================
class PMechDataset(Dataset):
    def __init__(self, csv_path, add_hs=True):
        super().__init__()
        self.add_hs = add_hs

        with open(csv_path, 'r', encoding='utf-8') as f:
            self.raw_lines = [line.strip() for line in f.readlines() if line.strip()]

    def __len__(self):
        return len(self.raw_lines)

    def __getitem__(self, idx):
        try:
            line = self.raw_lines[idx]
            reactant_smi, _, arrow_str = parse_pmech_row(line)
            aligned_mol, graph_data = process_mapped_smiles(reactant_smi, self.add_hs)

            if aligned_mol is None or graph_data.x.shape[0] == 0:
                return None

            target_delta = generate_delta_from_arrows(arrow_str, aligned_mol)
            graph_data.y = target_delta
            return graph_data

        except Exception:
            return None

def collate_fn(data_list):
    data_list = [item for item in data_list if item is not None]
    if not data_list:
        return None

    max_nodes = max(d.num_nodes for d in data_list)
    y_padded = []

    for d in data_list:
        pad_size = max_nodes - d.num_nodes
        padded = F.pad(d.y, (0, pad_size, 0, pad_size), value=0)
        y_padded.append(padded)

        # CRITICAL: Delete the raw y matrix so PyG doesn't try to collate it
        del d.y

    # Now PyG won't crash because the problematic multi-dimensional y matrices are gone
    batch = Batch.from_data_list(data_list)

    # Attach our manually padded batch to the final object
    batch.y_padded = torch.stack(y_padded)
    return batch


## Model

In [ ]:
# ==========================================
# 4. Architecture & Loss
# ==========================================
class MaskedFocalLoss(nn.Module):
    def __init__(self, weights=None, gamma=2, reduction='mean'):
        super().__init__()
        # FIX: Register weights as a buffer so it moves correctly with model.to(device)
        if weights is not None:
            self.register_buffer('weights', weights)
        else:
            self.weights = None

        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets, mask):
        mask_2d = mask.unsqueeze(1) * mask.unsqueeze(2)
        logits_flat = logits.view(-1, 3)
        targets_flat = targets.view(-1)
        mask_flat = mask_2d.view(-1)

        ce_loss = F.cross_entropy(logits_flat, targets_flat, weight=self.weights, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt)**self.gamma * ce_loss
        masked_loss = focal_loss * mask_flat

        if self.reduction == 'mean':
            num_active = mask_flat.sum().clamp(min=1.0)
            return masked_loss.sum() / num_active
        return masked_loss.view_as(targets)

class ReactionTransformer(nn.Module):
    def __init__(self, node_in=25, edge_in=6, hidden_dim=128, num_heads=4, num_layers=3, dropout=0.1):
        super().__init__()
        self.node_embedding = nn.Linear(node_in, hidden_dim)
        self.edge_embedding = nn.Linear(edge_in, hidden_dim)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(num_layers):
            self.convs.append(TransformerConv(
                in_channels=hidden_dim,
                out_channels=hidden_dim // num_heads,
                heads=num_heads,
                edge_dim=hidden_dim,
                dropout=dropout,
                concat=True
            ))
            self.norms.append(nn.LayerNorm(hidden_dim))

        self.mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 3)
        )

    def forward(self, x, edge_index, edge_attr, batch):
        h = self.node_embedding(x)
        e = self.edge_embedding(edge_attr)

        for conv, norm in zip(self.convs, self.norms):
            h_res = h
            h = conv(h, edge_index, edge_attr=e)
            h = F.dropout(h, p=0.1, training=self.training)
            h = norm(h + h_res)

        h_dense, mask = to_dense_batch(h, batch)
        B, N, D = h_dense.size()

        h_src = h_dense.unsqueeze(2).expand(B, N, N, D)
        h_dst = h_dense.unsqueeze(1).expand(B, N, N, D)
        pair_features = torch.cat([h_src, h_dst], dim=-1)

        logits = self.mlp(pair_features)

        # FIX: Enforce symmetric predictions (A->B must equal B->A)
        logits = (logits + logits.transpose(1, 2)) / 2

        return logits, mask


## Training

In [ ]:
# ==========================================
# 5. Training Engine
# ==========================================
def run_epoch(model, loader, optimizer, criterion, device, max_norm=1.0, is_train=True):
    model.train() if is_train else model.eval()

    total_loss = 0
    total_tp, total_fp, total_fn, total_topk = 0, 0, 0, 0

    for batch in loader:
        batch = batch.to(device)
        # Shift targets from [-1, 0, 1] to [0, 1, 2] for the CrossEntropy/Focal loss
        targets = batch.y_padded + 1

        with torch.set_grad_enabled(is_train):
            logits, mask = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            loss = criterion(logits, targets, mask)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
                optimizer.step()

        # Call the metric function on the REAL results from this batch
        tp, fp, fn, tk = get_mechanism_metrics(logits, targets, mask, k=3)

        total_loss += loss.item()
        total_tp += tp
        total_fp += fp
        total_fn += fn
        total_topk += tk

    avg_loss = total_loss / len(loader)
    # Average Top-K accuracy across all batches
    avg_topk = total_topk / len(loader)

    # Calculate global Precision/Recall for the whole epoch
    precision = total_tp / (total_tp + total_fp + 1e-8)
    recall = total_tp / (total_tp + total_fn + 1e-8)

    return avg_loss, recall, precision, avg_topk


def get_mechanism_metrics(logits, targets, mask, k=3):
    B, N, _, _ = logits.shape
    mask_2d = mask.unsqueeze(1) * mask.unsqueeze(2)
    triu_indices = torch.triu_indices(N, N, offset=1, device=logits.device)

    tp, fp, fn, topk_hits, total_rxns = 0, 0, 0, 0, 0

    for b in range(B):
        m = mask_2d[b]
        valid = m[triu_indices[0], triu_indices[1]]
        idx0, idx1 = triu_indices[0][valid], triu_indices[1][valid]
        if len(idx0) == 0: continue

        mol_logits = logits[b, idx0, idx1]
        mol_targets = targets[b, idx0, idx1]
        mol_preds = torch.argmax(mol_logits, dim=-1)

        # --- Standard P/R Logic ---
        is_rxn_target = (mol_targets != 1)
        is_rxn_pred = (mol_preds != 1)

        tp += (is_rxn_pred & is_rxn_target & (mol_preds == mol_targets)).sum().item()
        fp += (is_rxn_pred & (~is_rxn_target | (mol_preds != mol_targets))).sum().item()
        fn += (is_rxn_target & (~is_rxn_pred | (mol_preds != mol_targets))).sum().item()

        # --- Corrected Top-K (Center Identification) ---
        mol_total_rxns = is_rxn_target.sum().item()
        total_rxns += mol_total_rxns

        if mol_total_rxns > 0:
            probs = torch.softmax(mol_logits, dim=-1)
            # Rank by "Reaction Confidence" (Anything that isn't Class 1)
            rxn_probs = probs[:, 0] + probs[:, 2]

            k_to_use = min(k, len(rxn_probs))
            _, top_indices = torch.topk(rxn_probs, k_to_use)

            target_rxn_indices = torch.where(is_rxn_target)[0]
            for tr_idx in target_rxn_indices:
                # If the actual reaction center is in the Top K, it's a Top-K hit.
                if tr_idx in top_indices:
                    topk_hits += 1

    topk_acc = topk_hits / (total_rxns + 1e-8)
    return tp, fp, fn, topk_acc

## Tests

In [ ]:
def run_real_data_tests(num_samples=100):
    print(f"\n📂 Running Invariant Tests on Real PMechDB Data ({num_samples} samples)...\n")
    fold_dir = f"{CONFIG['base_path']}/fold0/reformatted"
    filepath = f"{fold_dir}/train.txt"

    print(f"\n📂 Testing Real Data from: {filepath}")

    if not os.path.exists(filepath):
        print(f"⚠️  SKIPPED: Path not found at {filepath}")
        return
    # Load the raw text lines
    with open(filepath, 'r', encoding='utf-8') as f:
        all_lines = [line.strip() for line in f.readlines() if line.strip()]

    # Randomly sample rows to avoid testing the same easy ones at the top of the file
    sample_lines = random.sample(all_lines, min(num_samples, len(all_lines)))

    passed = 0
    failed = 0

    for idx, line in enumerate(sample_lines):
        try:
            # 1. Test Parsing Invariants
            react, prod, arrows = parse_pmech_row(line)
            assert isinstance(react, str) and ">>" not in react, "Reactant string is malformed."
            assert isinstance(arrows, str), "Arrow string is missing or not a string."

            # 2. Test Processing & RDKit Invariants
            aligned_mol, graph_data = process_mapped_smiles(react, add_hs=CONFIG["add_explicit_hydrogens"])

            # RDKit might fail gracefully and return None for truly broken SMILES
            if aligned_mol is None:
                continue

            num_nodes = aligned_mol.GetNumAtoms()

            assert graph_data.x.shape[0] == num_nodes, f"Node feature count ({graph_data.x.shape[0]}) doesn't match atom count ({num_nodes})."
            assert graph_data.x.shape[1] == 25, "Node feature dimension changed from 25."

            if graph_data.edge_index.shape[1] > 0:
                assert graph_data.edge_attr.shape[1] == 6, "Edge feature dimension changed from 6."

            # 3. Test Delta Matrix Invariants
            delta = generate_delta_from_arrows(arrows, aligned_mol)

            assert delta.shape == (num_nodes, num_nodes), f"Delta matrix shape {delta.shape} doesn't match expected ({num_nodes}, {num_nodes})."
            assert torch.allclose(delta, delta.t()), "Delta matrix is not symmetric! A->B changes must equal B->A."

            # Check that delta values are strictly constrained to -1 (break), 0 (nothing), 1 (form/shift)
            unique_vals = torch.unique(delta)
            for val in unique_vals:
                assert val.item() in [-1, 0, 1], f"Illegal value {val.item()} found in delta matrix."

            passed += 1

        except AssertionError as e:
            print(f"❌ FAIL on sample {idx}: {line.strip()[:60]}...")
            print(f"   Details: {e}")
            failed += 1
        except Exception as e:
            print(f"⚠️ ERROR on sample {idx}: {line.strip()[:60]}...")
            print(f"   Crashed with {type(e).__name__}: {e}")
            failed += 1

    print("\n" + "="*45)
    print(f"🏁 Real Data Invariant Summary: {passed} Passed, {failed} Failed")
    print("="*45)


def run_architectural_sanity_checks():
    print("\n🧠 Running Architectural & Tensor Sanity Checks...\n")
    passed = 0
    failed = 0

    def check(test_name, condition, error_msg=""):
        nonlocal passed, failed
        try:
            assert condition, error_msg
            print(f"✅ PASS: {test_name}")
            passed += 1
        except AssertionError as e:
            print(f"❌ FAIL: {test_name}")
            if str(e): print(f"   Details: {e}")
            failed += 1
        except Exception as e:
            print(f"⚠️ ERROR: {test_name} crashed with {type(e).__name__}: {e}")
            failed += 1

    # Setup dummy data for testing
    # Graph 1: 3 nodes
    x1 = torch.randn(3, 25)
    edge_index1 = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long)
    edge_attr1 = torch.randn(4, 6)
    y1 = torch.randint(-1, 2, (3, 3)) # Targets: -1, 0, 1
    data1 = Data(x=x1, edge_index=edge_index1, edge_attr=edge_attr1, y=y1, num_nodes=3)

    # Graph 2: 5 nodes (to test padding)
    x2 = torch.randn(5, 25)
    edge_index2 = torch.tensor([[0, 3, 3, 4], [3, 0, 4, 3]], dtype=torch.long)
    edge_attr2 = torch.randn(4, 6)
    y2 = torch.randint(-1, 2, (5, 5))
    data2 = Data(x=x2, edge_index=edge_index2, edge_attr=edge_attr2, y=y2, num_nodes=5)

    # ---------------------------------------------------------
    # 6. Test Collation & Padding Logic
    # ---------------------------------------------------------
    batch = collate_fn([data1, data2])

    check("collate_fn: Batch generation",
          batch is not None and batch.num_graphs == 2,
          "Collate function failed to create a valid PyG Batch.")

    check("collate_fn: Target padding dimensions",
          batch.y_padded.shape == (2, 5, 5),
          f"Expected y_padded shape (2, 5, 5) to match max nodes, got {batch.y_padded.shape}")

    check("collate_fn: Zero-padding integrity",
          torch.all(batch.y_padded[0, 3:, :] == 0) and torch.all(batch.y_padded[0, :, 3:] == 0),
          "Padding did not fill the expanded matrix dimensions with zeros.")

    # ---------------------------------------------------------
    # 7. Test ReactionTransformer (Forward Pass & Symmetry)
    # ---------------------------------------------------------
    model = ReactionTransformer(node_in=25, edge_in=6, hidden_dim=16, num_heads=2, num_layers=2)
    model.eval()

    try:
        logits, mask = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)

        check("ReactionTransformer: Logit shape",
              logits.shape == (2, 5, 5, 3),
              f"Expected logits shape (2, 5, 5, 3), got {logits.shape}")

        check("ReactionTransformer: Mask shape",
              mask.shape == (2, 5),
              f"Expected mask shape (2, 5), got {mask.shape}")

        check("ReactionTransformer: Enforced logit symmetry",
              torch.allclose(logits, logits.transpose(1, 2), atol=1e-6),
              "Logits are not perfectly symmetric! A->B predictions differ from B->A.")

    except Exception as e:
        check("ReactionTransformer: Forward pass", False, f"Model forward pass crashed: {e}")

    # ---------------------------------------------------------
    # 8. Test MaskedFocalLoss
    # ---------------------------------------------------------
    criterion = MaskedFocalLoss(weights=torch.tensor([1.0, 1.0, 1.0]), gamma=2.0)
    targets = batch.y_padded + 1 # Shift -1,0,1 to 0,1,2 for CrossEntropy

    try:
        loss = criterion(logits, targets, mask)
        check("MaskedFocalLoss: Execution and scalar output",
              loss.dim() == 0 and not torch.isnan(loss),
              "Loss function failed to return a valid scalar.")
    except Exception as e:
        check("MaskedFocalLoss: Execution", False, f"Loss calculation crashed: {e}")

    # ---------------------------------------------------------
    # 9. Test Metric Calculation Logic (Addressing Doubling)
    # ---------------------------------------------------------
    fake_logits = torch.zeros(1, 2, 2, 3)

    # FIX: Set the default prediction to Class 1 ("Nothing") so argmax
    # doesn't interpret the 0.0s as Class 0 ("Break") on the diagonals.
    fake_logits[:, :, :, 1] = 1.0

    # Predict form (class 2) between 0 and 1
    fake_logits[0, 0, 1, 2] = 10.0
    fake_logits[0, 1, 0, 2] = 10.0

    # Target is break (class 0) between 0 and 1
    fake_targets = torch.tensor([[[1, 0], [0, 1]]])
    fake_mask = torch.tensor([[True, True]])

    # Add the 4th variable (we'll call it _ since we aren't testing Top-K in this specific check)
    tp, fp, fn, _ = get_mechanism_metrics(fake_logits, fake_targets, fake_mask)

    check("get_mechanism_metrics: FP/FN logic and symmetry halving",
          tp == 0.0 and fp == 1.0 and fn == 1.0,
          f"Expected TP=0, FP=1, FN=1. Got TP={tp}, FP={fp}, FN={fn}. Check matrix halving logic.")

    print("\n" + "="*30)
    print(f"🏁 Architecture Sanity Check Summary: {passed} Passed, {failed} Failed")
    print("="*30)


def run_real_data_edge_case_tests(num_samples=20):
    print(f"\n🧨 Running Advanced Edge Case Tests on Mutated Real Data...\n")
    passed = 0
    failed = 0

    def check(test_name, condition, error_msg=""):
        nonlocal passed, failed
        try:
            assert condition, error_msg
            print(f"✅ PASS: {test_name}")
            passed += 1
        except AssertionError as e:
            print(f"❌ FAIL: {test_name}")
            if str(e): print(f"   Details: {e}")
            failed += 1
        except Exception as e:
            print(f"⚠️ ERROR: {test_name} crashed with {type(e).__name__}: {e}")
            failed += 1

    fold_dir = f"{CONFIG['base_path']}/fold0/reformatted"
    filepath = f"{fold_dir}/train.txt"

    print(f"\n📂 Testing Real Data from: {filepath}")

    if not os.path.exists(filepath):
        print(f"⚠️  SKIPPED: Path not found at {filepath}")
        return
    # Load the raw text lines
    with open(filepath, 'r', encoding='utf-8') as f:
        all_lines = [line.strip() for line in f.readlines() if line.strip()]


    sample_lines = random.sample(all_lines, min(num_samples, len(all_lines)))

    for idx, raw_line in enumerate(sample_lines):
        # We need the cleanly parsed parts to create our mutations
        try:
            reader = csv.reader([raw_line])
            parts = next(reader)
            equation = parts[0]
            arrows = parts[1] if len(parts) > 1 else ""
            react_smi = equation.split(">>")[0]
            mol = Chem.MolFromSmiles(react_smi)
        except Exception:
            continue # Skip rows that are genuinely broken before we even mutate them

        if mol is None:
            continue

        # ---------------------------------------------------------
        # 1. parse_pmech_row: Mutated Malformed Data Handling
        # ---------------------------------------------------------
        # Mutate 1: Strip the arrows entirely (simulate missing column)
        try:
            parse_pmech_row(equation)
            check(f"Row {idx} | parse_pmech: Missing arrows column", False, "Failed to raise ValueError on missing arrows.")
        except ValueError:
            check(f"Row {idx} | parse_pmech: Missing arrows column", True)

        # Mutate 2: Strip the >> operator from the real equation
        bad_equation = equation.replace(">>", "-")
        bad_csv_row = f'{bad_equation},"{arrows}"'
        try:
            parse_pmech_row(bad_csv_row)
            check(f"Row {idx} | parse_pmech: Missing >> operator", False, "Failed to raise ValueError on missing >>.")
        except ValueError:
            check(f"Row {idx} | parse_pmech: Missing >> operator", True)

        # ---------------------------------------------------------
        # 2. generate_delta_from_arrows: Dynamic Edge Cases
        # ---------------------------------------------------------
        # Test Phantom Map Numbers on a real molecule
        try:
            delta_phantom = generate_delta_from_arrows("9999=; 9999=9998", mol)
            check(f"Row {idx} | generate_delta: Phantom map numbers",
                  delta_phantom.sum().item() == 0,
                  "Did not safely ignore map numbers that don't exist in the real molecule.")
        except Exception as e:
            check(f"Row {idx} | generate_delta: Phantom map numbers", False, f"Crashed on phantom maps: {e}")

        # Test Empty/Null Arrow Strings on real molecule
        delta_empty = generate_delta_from_arrows("   ", mol)
        check(f"Row {idx} | generate_delta: Empty arrow string",
              delta_empty.sum().item() == 0,
              "Failed to return a clean zero matrix for empty arrow strings.")

        # ---------------------------------------------------------
        # 3. process_mapped_smiles: Dynamic Hydrogen Tracking
        # ---------------------------------------------------------
        # Process WITHOUT adding explicit Hs
        _, data_no_h = process_mapped_smiles(react_smi, add_hs=False)
        nodes_no_h = data_no_h.x.shape[0]

        # Process WITH adding explicit Hs
        _, data_with_h = process_mapped_smiles(react_smi, add_hs=True)
        nodes_with_h = data_with_h.x.shape[0]

        check(f"Row {idx} | process_mapped: Hydrogen logic",
              nodes_with_h >= nodes_no_h,
              f"Explicit H count ({nodes_with_h}) is less than implicit count ({nodes_no_h}).")

    # ---------------------------------------------------------
    # 4. process_mapped_smiles: Garbage Inputs (Static)
    # ---------------------------------------------------------
    # This must remain static. You can't pull a garbage SMILES from a clean database.
    try:
        process_mapped_smiles("THIS_IS_NOT_A_SMILES")
        check("Static | process_mapped: Invalid SMILES", False, "Failed to raise ValueError on garbage SMILES.")
    except ValueError:
        check("Static | process_mapped: Invalid SMILES", True)

    print("\n" + "="*45)
    print(f"🏁 Mutated Edge Case Summary: {passed} Passed, {failed} Failed")
    print("="*45)

if __name__ == "__main__":
    # You can call this alongside your other test suites
    run_real_data_tests()
    run_architectural_sanity_checks()
    run_real_data_edge_case_tests()


📂 Running Invariant Tests on Real PMechDB Data (100 samples)...


📂 Testing Real Data from: /content/drive/MyDrive/pmechrp_datasets/two_stage_models/mixed_splits/fold0/reformatted/train.txt

🏁 Real Data Invariant Summary: 100 Passed, 0 Failed

🧠 Running Architectural & Tensor Sanity Checks...

✅ PASS: collate_fn: Batch generation
✅ PASS: collate_fn: Target padding dimensions
✅ PASS: collate_fn: Zero-padding integrity
✅ PASS: ReactionTransformer: Logit shape
✅ PASS: ReactionTransformer: Mask shape
✅ PASS: ReactionTransformer: Enforced logit symmetry
✅ PASS: MaskedFocalLoss: Execution and scalar output
✅ PASS: get_mechanism_metrics: FP/FN logic and symmetry halving

🏁 Architecture Sanity Check Summary: 8 Passed, 0 Failed

🧨 Running Advanced Edge Case Tests on Mutated Real Data...


📂 Testing Real Data from: /content/drive/MyDrive/pmechrp_datasets/two_stage_models/mixed_splits/fold0/reformatted/train.txt
✅ PASS: Row 0 | parse_pmech: Missing arrows column
✅ PASS: Row 0 | parse_pmech: Miss

[09:00:43] SMILES Parse Error: syntax error while parsing: THIS_IS_NOT_A_SMILES
[09:00:43] SMILES Parse Error: check for mistakes around position 1:
[09:00:43] THIS_IS_NOT_A_SMILES
[09:00:43] ^
[09:00:43] SMILES Parse Error: Failed parsing SMILES 'THIS_IS_NOT_A_SMILES' for input: 'THIS_IS_NOT_A_SMILES'


## Execution

In [ ]:
def plot_metrics(history, save_path='training_metrics.png'):
    epochs = range(1, len(history['train_loss']) + 1)
    # Increased figure size to accommodate 4 plots
    plt.figure(figsize=(20, 5))

    # 1. Loss
    plt.subplot(1, 4, 1)
    plt.plot(epochs, history['train_loss'], label='Train')
    plt.plot(epochs, history['val_loss'], label='Val')
    plt.title('Focal Loss')
    plt.xlabel('Epoch')
    plt.legend()

    # 2. Precision & Recall
    plt.subplot(1, 4, 2)
    plt.plot(epochs, history['val_recall'], label='Recall', color='orange')
    plt.plot(epochs, history['val_precision'], label='Precision', color='green')
    plt.title('P/R Metrics')
    plt.xlabel('Epoch')
    plt.legend()

    # 3. Balanced F1-Score
    plt.subplot(1, 4, 3)
    plt.plot(epochs, history['val_f1'], label='F1 Score', color='purple', linewidth=2)
    plt.title('F1 Score (P/R Balance)')
    plt.xlabel('Epoch')
    plt.legend()

    # 4. Top-3 Accuracy
    plt.subplot(1, 4, 4)
    plt.plot(epochs, history['val_top3'], label='Top-3 Acc', color='red', linestyle='--')
    plt.title('Top-3 Accuracy')
    plt.xlabel('Epoch')
    plt.legend()

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

if __name__ == "__main__":
    # --- 1. Data Setup ---
    FOLD_DIR = f"{CONFIG['base_path']}/fold0/reformatted"
    train_loader = DataLoader(
        PMechDataset(f"{FOLD_DIR}/train.txt", add_hs=CONFIG["add_explicit_hydrogens"]),
        batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
    )
    val_loader = DataLoader(
        PMechDataset(f"{FOLD_DIR}/val.txt", add_hs=CONFIG["add_explicit_hydrogens"]),
        batch_size=CONFIG["batch_size"], collate_fn=collate_fn
    )

    # --- 2. Initialize Model & Optimizer ---
    DEVICE = torch.device(CONFIG["device"])
    model = ReactionTransformer(
        node_in=25, edge_in=6, hidden_dim=CONFIG["hidden_dim"],
        num_heads=CONFIG["num_heads"], num_layers=CONFIG["num_layers"], dropout=CONFIG["dropout"]
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])

    # --- 3. Scheduler ---
    # We monitor 'v_loss'. If it doesn't improve for 5 epochs (patience),
    # we multiply LR by 0.5 (factor).
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5
    )

    weights = torch.tensor(CONFIG["class_weights"], dtype=torch.float).to(DEVICE)
    criterion = MaskedFocalLoss(weights=weights, gamma=2.0).to(DEVICE)

    # --- 4. History & Early Stopping Setup ---
    history = {
        'train_loss': [], 'val_loss': [],
        'val_recall': [], 'val_precision': [],
        'val_f1': [], 'val_top3': []
    }

    best_f1 = 0.0
    early_stop_patience = 12  # Give the scheduler time to work before killing the run
    trigger_times = 0

    print(f"🚀 Starting POC Training with LR Scheduler (Patience: {early_stop_patience})")

    # --- 5. Main Training Loop ---
    for epoch in range(CONFIG["num_epochs"]):
        # Run Epoch
        t_loss, t_rec, t_pre, t_tk = run_epoch(model, train_loader, optimizer, criterion, DEVICE, CONFIG["max_grad_norm"], is_train=True)
        v_loss, v_rec, v_pre, v_tk = run_epoch(model, val_loader, optimizer, criterion, DEVICE, is_train=False)

        # Step the Scheduler based on Validation Loss
        scheduler.step(v_loss)

        # Get current LR for logging
        current_lr = optimizer.param_groups[0]['lr']

        # F1-Score calculation
        v_f1 = 2 * (v_pre * v_rec) / (v_pre + v_rec + 1e-8)

        # Log History
        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['val_recall'].append(v_rec)
        history['val_precision'].append(v_pre)
        history['val_f1'].append(v_f1)
        history['val_top3'].append(v_tk)

        plot_metrics(history)

        print(f"Ep {epoch+1:02d} | LR: {current_lr:.2e} | Val Loss: {v_loss:.4f} | Rec: {v_rec:.3f} | Pre: {v_pre:.3f} | F1: {v_f1:.3f} | Top3: {v_tk:.3f}")

        # --- Checkpointing & Early Stopping ---
        if v_f1 > best_f1:
            best_f1 = v_f1
            trigger_times = 0
            torch.save(model.state_dict(), 'poc_best_model.pt')
            print(f"⭐ New Best F1! Checkpoint saved.")
        else:
            trigger_times += 1
            print(f"◽ No F1 improvement. Patience: {trigger_times}/{early_stop_patience}")

        if trigger_times >= early_stop_patience:
            print(f"\n🛑 Early Stopping triggered. Best F1: {best_f1:.4f}")
            break

    print("\n🏁 Training Complete.")

🚀 Starting POC Training with LR Scheduler (Patience: 12)


[09:00:55] WARNING: not removing hydrogen atom without neighbors
[09:00:55] WARNING: not removing hydrogen atom without neighbors
[09:00:56] WARNING: not removing hydrogen atom without neighbors
[09:00:56] WARNING: not removing hydrogen atom without neighbors
[09:00:59] WARNING: not removing hydrogen atom without neighbors
[09:01:04] WARNING: not removing hydrogen atom without neighbors
[09:01:04] WARNING: not removing hydrogen atom without neighbors
[09:01:11] WARNING: not removing hydrogen atom without neighbors
[09:01:13] Explicit valence for atom # 5 Br, 2, is greater than permitted
[09:01:20] Explicit valence for atom # 0 B, 4, is greater than permitted
[09:01:22] WARNING: not removing hydrogen atom without neighbors
[09:01:23] WARNING: not removing hydrogen atom without neighbors
[09:01:23] WARNING: not removing hydrogen atom without neighbors
[09:01:24] WARNING: not removing hydrogen atom without neighbors
[09:01:28] WARNING: not removing hydrogen atom without neighbors
[09:01:3

Ep 01 | LR: 1.00e-04 | Val Loss: 0.0078 | Rec: 0.089 | Pre: 0.502 | F1: 0.152 | Top3: 0.478
⭐ New Best F1! Checkpoint saved.


[09:02:26] Explicit valence for atom # 5 Br, 2, is greater than permitted
[09:02:28] WARNING: not removing hydrogen atom without neighbors
[09:02:33] WARNING: not removing hydrogen atom without neighbors
[09:02:35] WARNING: not removing hydrogen atom without neighbors
[09:02:39] WARNING: not removing hydrogen atom without neighbors
[09:02:39] Explicit valence for atom # 10 Br, 2, is greater than permitted
[09:02:40] WARNING: not removing hydrogen atom without neighbors
[09:02:43] WARNING: not removing hydrogen atom without neighbors
[09:02:44] WARNING: not removing hydrogen atom without neighbors
[09:02:46] WARNING: not removing hydrogen atom without neighbors
[09:02:50] Explicit valence for atom # 0 B, 4, is greater than permitted
[09:02:50] WARNING: not removing hydrogen atom without neighbors
[09:02:52] WARNING: not removing hydrogen atom without neighbors
[09:02:53] WARNING: not removing hydrogen atom without neighbors
[09:03:04] WARNING: not removing hydrogen atom without neighbor

Ep 02 | LR: 1.00e-04 | Val Loss: 0.0051 | Rec: 0.380 | Pre: 0.316 | F1: 0.345 | Top3: 0.642
⭐ New Best F1! Checkpoint saved.


...

Ep 48 | LR: 5.00e-05 | Val Loss: 0.0018 | Rec: 0.898 | Pre: 0.460 | F1: 0.608 | Top3: 0.899
◽ No F1 improvement. Patience: 1/12


[10:17:23] Explicit valence for atom # 5 Br, 2, is greater than permitted
[10:17:29] Explicit valence for atom # 10 Br, 2, is greater than permitted
[10:17:31] WARNING: not removing hydrogen atom without neighbors
[10:17:34] WARNING: not removing hydrogen atom without neighbors
[10:17:41] Explicit valence for atom # 21 B, 4, is greater than permitted
[10:17:44] Explicit valence for atom # 4 Sn, 5, is greater than permitted
[10:17:51] WARNING: not removing hydrogen atom without neighbors
[10:17:55] WARNING: not removing hydrogen atom without neighbors
[10:17:59] WARNING: not removing hydrogen atom without neighbors
[10:18:01] WARNING: not removing hydrogen atom without neighbors
[10:18:04] WARNING: not removing hydrogen atom without neighbors
[10:18:07] WARNING: not removing hydrogen atom without neighbors
[10:18:08] WARNING: not removing hydrogen atom without neighbors
[10:18:08] Explicit valence for atom # 5 Br, 2, is greater than permitted
[10:18:08] WARNING: not removing hydrogen at

Ep 49 | LR: 5.00e-05 | Val Loss: 0.0018 | Rec: 0.890 | Pre: 0.471 | F1: 0.616 | Top3: 0.893
◽ No F1 improvement. Patience: 2/12


[10:19:00] WARNING: not removing hydrogen atom without neighbors
[10:19:00] WARNING: not removing hydrogen atom without neighbors
[10:19:05] WARNING: not removing hydrogen atom without neighbors
[10:19:05] WARNING: not removing hydrogen atom without neighbors
[10:19:08] Explicit valence for atom # 4 Sn, 5, is greater than permitted
[10:19:16] WARNING: not removing hydrogen atom without neighbors
[10:19:18] Explicit valence for atom # 10 Br, 2, is greater than permitted
[10:19:20] WARNING: not removing hydrogen atom without neighbors
[10:19:24] Explicit valence for atom # 5 Br, 2, is greater than permitted
[10:19:29] WARNING: not removing hydrogen atom without neighbors
[10:19:30] WARNING: not removing hydrogen atom without neighbors
[10:19:30] WARNING: not removing hydrogen atom without neighbors
[10:19:33] WARNING: not removing hydrogen atom without neighbors
[10:19:33] WARNING: not removing hydrogen atom without neighbors
[10:19:44] WARNING: not removing hydrogen atom without neighbo

Ep 50 | LR: 5.00e-05 | Val Loss: 0.0018 | Rec: 0.896 | Pre: 0.457 | F1: 0.605 | Top3: 0.899
◽ No F1 improvement. Patience: 3/12

🏁 Training Complete.
